# P09 — Harmonici i zvuk: djelomični zbrojevi Fourierova reda i Gibbsova pojava

Bilježnica uz predavanje P09 i **poglavlje 9 udžbenika *Signali i sustavi*** (Fourierov red i
Fourierova transformacija). Pravokutni, pilasti i trokutni val osnovne frekvencije
$f_0 = 220$ Hz gradimo zbrajanjem harmonika. Klizačem se bira broj harmonika $N$: bilježnica
crta djelomični zbroj $S_N$ uz sam val i spektar (linije $b_k$), a zbroj pušta kao zvuk, pa se
razlika između sinusoide i vala s više harmonika i čuje. Odjeljak o Gibbsovoj pojavi uspoređuje
vrh zbroja uz skok s granicom iz poglavlja 9. Zadnji pokus zbraja iste harmonike s drukčijim
fazama: amplitudni spektar je isti, a oblik vala drukčiji; zvuk je sličan, ali ne nužno
istovjetan.

**Kako pokrenuti.** Ćelije pokrenite redom (Shift + Enter) ili izbornikom *Run → Run All
Cells*. Instalaciju Pythona i JupyterLaba opisuje `README.md` repozitorija. Bilježnica radi bez
interneta, a sve zvukove sama sintetizira. Za slušanje trebaju zvučnici ili slušalice; bez njih
rade grafovi i ispis. Ako se klizač ne prikaže, funkciju pokusa pozovite izravno (odjeljak 3).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator
from scipy.optimize import minimize_scalar
from scipy.special import sici
from IPython.display import Audio, display
import ipywidgets as widgets

BOJA_S = "#840B55"                     # bordo: djelomični zbroj S_N
SIVA, TAMNA = "#5B6770", "#222222"
# Krivulje se razlikuju vrstom i debljinom crte, pa su čitljive i crno-bijelo: val je debela
# svijetla crta, zbrojevi su tanke tamne crte različitih vrsta.
CRTE = ["-", "--", "-.", ":"]

# decimalni zarez i pravi minus na osima
zarez = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ",").replace("-", "−"))
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3})

## Postavke

U poglavlju 9 valovi imaju period $T_0 = 2\pi$ s ($\omega_0 = 1$ rad/s). Takav val bio bi
preniske frekvencije da se čuje, pa ga ovdje ubrzamo na $f_0 = 220$ Hz (ton A3):
$T_0 = 1/f_0$, $\omega_0 = 2\pi f_0$. Koeficijenti $b_k$ ne ovise o $f_0$. Drugu osnovnu
frekvenciju ili drugi raspon klizača zadajte u ćeliji ispod, pa pokrenite sve ćelije.

In [ ]:
# ===== POSTAVKE (mijenjajte po želji, pa Run → Run All Cells) =====
F0 = 220.0               # Hz: osnovna frekvencija f₀ (ton A3)
FS = 44100               # Hz: frekvencija uzorkovanja zvuka f_s
TRAJANJE = 1.5           # s: trajanje svakog zvuka
GLASNOCA = 0.5           # isti faktor za sve zvukove: mijenja se boja tona, a ne glasnoća
N_MAX = 51               # najveći broj harmonika N na klizaču
N_POCETNI = 7            # N na klizaču nakon pokretanja
VAL_POCETNI = "pravokutni"
N_SLIKA = (1, 7, 21)     # N na slici u odjeljku 2 i u nizu zvukova (kao slika u poglavlju 9)
N_ISPIS_MIN = 11         # vrh i nadvišenje ispisuju se za N ≥ 11, kao u poglavlju 9
N_GIBBS_MAX = 101        # najveći N na slici nadvišenja (odjeljak 4)
N_FAZE = 21              # broj harmonika u pokusu s fazama (odjeljak 6)
SJEME = 2026             # sjeme slučajnih faza: isti rezultat pri svakom pokretanju

# Iz knjige (poglavlje 9, Gibbsova pojava): nadvišenje vrha iznad 1 za N = 11, 21 i 51,
# granica vrha (2/π)·Si(π), granica nadvišenja i nadvišenje u postocima skoka (odjeljak 4)
NADVISENJE_KNJIGA = {11: 0.1813, 21: 0.1797, 51: 0.1791}
GRANICA_KNJIGA = 1.179
NADVISENJE_GRANICA_KNJIGA = 0.1790
POSTO_KNJIGA = 8.9

## 1. Tri vala i njihovi koeficijenti

Sva tri vala imaju period $T_0$, visinu $\pm 1$ i srednju vrijednost nula. Neparni su, pa
njihov Fourierov red (poglavlje 9) ima samo sinuse:

$$x(t) = \sum_{k=1}^{\infty} b_k \sin k\omega_0 t ,\qquad
S_N(t) = \sum_{k=1}^{N} b_k \sin k\omega_0 t .$$

Djelomični zbroj $S_N$ sadrži harmonike do $N$-tog; $k$-ti harmonik ima frekvenciju $k f_0$.

| val | jedan period | $b_k$ |
|---|---|---|
| pravokutni | $+1$ na $(0,\ T_0/2)$, $-1$ na $(T_0/2,\ T_0)$ | $\dfrac{4}{k\pi}$ za neparni $k$, $0$ za parni |
| pilasti | raste jednoliko od $-1$ do $+1$ na $(-T_0/2,\ T_0/2)$ | $\dfrac{2\,(-1)^{k+1}}{k\pi}$ |
| trokutni | $\frac{2}{\pi}\arcsin(\sin\omega_0 t)$, vrhovi $\pm1$ u $T_0/4$ i $3T_0/4$ | $\dfrac{8\,(-1)^{(k-1)/2}}{k^2\pi^2}$ za neparni $k$, $0$ za parni |

Pravokutni val je val $\mathrm{sgn}(\sin\omega_0 t)$ iz poglavlja 9. Pilasti val je pilasti
val iz poglavlja 9 ($x = t$ na $(-\pi, \pi)$) podijeljen s $\pi$, da mu visina bude $\pm 1$.
Kako brzo koeficijenti padaju s $k$, opisuje pravilo o glatkoći i padu koeficijenata u
poglavlju 9.

Funkcije u ćeliji ispod računaju koeficijente, val, djelomični zbroj, vrh zbroja uz skok,
zvuk i ispis.

In [ ]:
VALOVI = ("pravokutni", "pilasti", "trokutni")
NAZIVI = {"pravokutni": "pravokutni val", "pilasti": "pilasti val", "trokutni": "trokutni val"}
SKOK = 2.0               # skok pravokutnog vala: s −1 na +1
RUB_ZVUKA = 0.02         # s: meki početak i kraj zvuka (bez klika)
STANKA = 0.3             # s: stanka između zvukova u nizu


def broj(v, decimala=2):
    """Broj s decimalnim zarezom i pravim minusom (−0,50); −0,00 piše se 0,00."""
    tekst = f"{v:.{decimala}f}"
    if float(tekst) == 0:
        tekst = tekst.lstrip("-")
    return tekst.replace(".", ",").replace("-", "−")


def herci(f):
    """Frekvencija u Hz: cijeli broj bez decimala (220), inače jedna decimala (27,5)."""
    return broj(f, 0) if float(f).is_integer() else broj(f, 1)


def koeficijent(val, k):
    """Koeficijent b_k vala visine ±1 (tablica iznad)."""
    if val == "pravokutni":
        return 0.0 if k % 2 == 0 else 4 / (k * np.pi)
    if val == "pilasti":
        return 2 * (-1) ** (k + 1) / (k * np.pi)
    if val == "trokutni":
        return 0.0 if k % 2 == 0 else 8 * (-1) ** ((k - 1) // 2) / (k * np.pi) ** 2
    raise ValueError(f"nepoznat val „{val}”; mogući su: {', '.join(VALOVI)}")


def valni_oblik(val, t, f0=F0):
    """Sam val x(t) visine ±1."""
    t = np.asarray(t, dtype=float)
    if val == "pravokutni":
        return np.sign(np.sin(2 * np.pi * f0 * t))
    if val == "pilasti":
        return 2 * ((f0 * t + 0.5) % 1) - 1
    if val == "trokutni":
        return 2 / np.pi * np.arcsin(np.sin(2 * np.pi * f0 * t))
    raise ValueError(f"nepoznat val „{val}”; mogući su: {', '.join(VALOVI)}")


def zbroj(val, N, t, faze=None, f0=F0):
    """Djelomični zbroj S_N(t) = Σ b_k sin(kω₀t + φ_k), k = 1 … N; bez faza su sve φ_k = 0."""
    t = np.asarray(t, dtype=float)
    s = np.zeros_like(t)
    for k in range(1, N + 1):
        b = koeficijent(val, k)
        if b:
            fi = 0.0 if faze is None else faze[k - 1]
            s = s + b * np.sin(2 * np.pi * k * f0 * t + fi)
    return s


def vrh(val, N, f0=F0):
    """Vrijednost prvog vrha S_N pravokutnog vala nakon skoka u t = 0, za N ≥ N_ISPIS_MIN.
    Traži se najveći uzorak na prvoj četvrtini perioda, pa se vrh dotjera (minimize_scalar)."""
    if val != "pravokutni":
        raise ValueError("vrh se ovdje računa samo za pravokutni val")
    if N < N_ISPIS_MIN:
        raise ValueError(f"vrh se računa za N ≥ {N_ISPIS_MIN}; za manji N pogledajte graf")
    T0 = 1 / f0
    t = np.linspace(0, T0 / 4, 40 * N + 2)[1:]
    i = int(np.argmax(zbroj(val, N, t, f0=f0)))
    lijevo, desno = t[max(i - 1, 0)], t[min(i + 1, len(t) - 1)]
    r = minimize_scalar(lambda u: -float(zbroj(val, N, u, f0=f0)), bounds=(lijevo, desno),
                        method="bounded", options={"xatol": 1e-12 * T0})
    return -float(r.fun)


def gibbs_granica():
    """Granica vrha S_N pravokutnog vala kad N → ∞: (2/π)·Si(π) (poglavlje 9)."""
    return 2 / np.pi * float(sici(np.pi)[0])


def najvisi_u_zvuku(N, f0=F0, fs=FS):
    """Najviši harmonik u zvuku: samo harmonici ispod f_s/2 (viši bi se čuli na krivoj
    frekvenciji, poglavlje 12)."""
    return min(N, int(np.ceil(fs / 2 / f0)) - 1)


def zvuk(val, N, faze=None, glasnoca=GLASNOCA, f0=F0, fs=FS, trajanje=TRAJANJE):
    """S_N kao zvuk: uzorci uz f_s, harmonici ispod f_s/2, isti faktor glasnoće za svaki N i
    meki početak i kraj (bez klika); vrijednosti u rasponu −1 … 1."""
    t = np.arange(int(trajanje * fs)) / fs
    s = glasnoca * zbroj(val, najvisi_u_zvuku(N, f0, fs), t, faze, f0)
    rub = int(RUB_ZVUKA * fs)
    s[:rub] *= np.linspace(0, 1, rub)
    s[-rub:] *= np.linspace(1, 0, rub)
    return np.clip(s, -1, 1)


def niz_zvukova(val, niz_N):
    """Zvukovi S_N za N iz niza, jedan za drugim, sa stankom između njih."""
    tisina = np.zeros(int(STANKA * FS))
    return np.concatenate([d for N in niz_N for d in (zvuk(val, N), tisina)])


def opis_niza(val, niz_N):
    """Redak ispisa uz niz zvukova."""
    return f"{NAZIVI[val]}: S_N redom za N = {', '.join(str(N) for N in niz_N)}"


def opis(val, N, f0=F0):
    """Retci ispisa za val i N: harmonici u zbroju, a za pravokutni val i N ≥ N_ISPIS_MIN
    prvi vrh nakon skoka i nadvišenje."""
    harmonici = [k for k in range(1, N + 1) if koeficijent(val, k)]
    retci = [f"{NAZIVI[val]}, N = {N}: harmonika u zbroju {len(harmonici)}, najviši na "
             f"{herci(harmonici[-1] * f0)} Hz (f₀ = {herci(f0)} Hz)"]
    if najvisi_u_zvuku(N, f0) < N:
        retci.append(f"u zvuku su samo harmonici do {najvisi_u_zvuku(N, f0)} "
                     f"(ispod f_s/2 = {herci(FS / 2)} Hz)")
    if val == "pravokutni":
        if N >= N_ISPIS_MIN:
            v = vrh(val, N, f0)
            retci.append(f"prvi vrh nakon skoka: S_N = {broj(v, 4)}")
            retci.append(f"nadvišenje iznad razine 1: {broj(v - 1, 4)}, "
                         f"tj. {broj(100 * (v - 1) / SKOK, 1)} % skoka (skok je 2)")
        else:
            retci.append(f"vrh i nadvišenje ispisuju se za N ≥ {N_ISPIS_MIN}; "
                         "za manji N oblik zbroja pokazuje graf")
    elif val == "pilasti":
        retci.append("uz skok pilastog vala zbroj također preskače val; brojke vrha ispisuju se "
                     "samo za pravokutni val (odjeljak 4)")
    else:
        retci.append("trokutni val nema skoka, pa nema ni Gibbsove pojave")
    return retci


def tablica_gibbs():
    """Retci tablice: vrh i nadvišenje pravokutnog vala za N iz NADVISENJE_KNJIGA (poglavlje 9)
    i granica kad N → ∞."""
    retci = ["   N    vrh S_N   nadvišenje   % skoka   knjiga: nadvišenje"]
    for N, knjiga in NADVISENJE_KNJIGA.items():
        v = vrh("pravokutni", N)
        retci.append(f"{N:>4}   {broj(v, 4):>7}   {broj(v - 1, 4):>10}   "
                     f"{broj(100 * (v - 1) / SKOK, 1):>5} %   "
                     f"{broj(knjiga, 4):>18}")
    g = gibbs_granica()
    retci.append(f"granica (teorija, N → ∞): vrh (2/π)·Si(π) = {broj(g, 3)}, nadvišenje "
                 f"{broj(g - 1, 4)}, tj. {broj(100 * (g - 1) / SKOK, 1)} % skoka; "
                 f"knjiga: {broj(GRANICA_KNJIGA, 3)}, {broj(NADVISENJE_GRANICA_KNJIGA, 4)}, "
                 f"{broj(POSTO_KNJIGA, 1)} %")
    return retci


for redak in opis(VAL_POCETNI, N_POCETNI):
    print(redak)

## 2. Djelomični zbrojevi za nekoliko N

Gore je za svaki val djelomični zbroj $S_N$ za $N$ = 1, 7 i 21 (kao slika djelomičnih zbrojeva
u poglavlju 9) uz sam val (debela svijetla crta), na dva perioda. Dolje je spektar: linije $b_k$
do $k = 21$. Uz $N = 1$ zbroj je sinusoida osnovne frekvencije, a svaki novi harmonik dodaje
finije pojedinosti oblika.

In [ ]:
def spektar(ax, val, do_k, u_zbroju=None):
    # linije b_k za k = 1 … do_k; linije izvan zbroja (k > u_zbroju) svijetle i tanke
    k = np.arange(1, do_k + 1)
    b = np.array([koeficijent(val, kk) for kk in k])
    u_zbroju = do_k if u_zbroju is None else u_zbroju
    for dio, boja, lw, oznaka in ((k <= u_zbroju, TAMNA, 1.6, "o"), (k > u_zbroju, SIVA, 0.8, ".")):
        if dio.any():
            ax.vlines(k[dio], 0, b[dio], color=boja, lw=lw)
            ax.plot(k[dio], b[dio], oznaka, color=boja, mfc="white", ms=5)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_xlim(0, do_k + 1)
    ax.set_ylim(-0.8, 1.4)
    ax.set_xlabel("harmonik $k$ (frekvencija $k f_0$)")
    ax.set_ylabel("$b_k$")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, steps=[1, 2, 5, 10]))
    ax.yaxis.set_major_formatter(zarez)


t = np.linspace(0, 2 / F0, 2001)
fig, osi = plt.subplots(2, 3, figsize=(13, 6.4))
for j, val in enumerate(VALOVI):
    ax = osi[0, j]
    ax.plot(1e3 * t, valni_oblik(val, t), color=SIVA, lw=5, alpha=0.45, label="val $x(t)$")
    for i, N in enumerate(N_SLIKA):
        ax.plot(1e3 * t, zbroj(val, N, t), color=TAMNA, lw=1.0 + 0.4 * i,
                ls=CRTE[(len(N_SLIKA) - i) % len(CRTE)], label=f"$S_{{{N}}}$")
    ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel("vrijeme $t$ / ms")
    ax.set_title(NAZIVI[val], loc="left", fontsize=10)
    ax.xaxis.set_major_formatter(zarez)
    ax.yaxis.set_major_formatter(zarez)
    spektar(osi[1, j], val, N_SLIKA[-1])
osi[0, 2].legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)   # izvan krivulja
fig.tight_layout()
plt.show()

## 3. Klizač: broj harmonika N i zvuk

Odaberite val i klizačem broj harmonika $N$ (od 1 do 51). Lijevo gore je $S_N$ uz val na dva
perioda, desno gore uvećani dio uz skok (kod trokutnog vala uz vrh), a dolje spektar: tamne
linije $b_k$ su u zbroju, svijetle još nisu. Ispod grafa su broj harmonika i najviša
frekvencija u zbroju, a za pravokutni val i $N \ge 11$ i prvi vrh nakon skoka i nadvišenje. Za
manji $N$ oblik zbroja pokazuje graf.

Na kraju se $S_N$ pušta kao zvuk. Uz $N = 1$ čuje se čista sinusoida frekvencije $f_0$. Dodavanjem
harmonika visina tona ostaje ista, a mijenja se boja tona. Svi zvukovi imaju isti faktor
glasnoće (`GLASNOCA`).

Ako se klizač ne prikaže, pozovite funkciju izravno, npr. `pokus("pilasti", 15)`.

In [ ]:
def pokus(val=VAL_POCETNI, N=N_POCETNI):
    T0 = 1 / F0
    t = np.linspace(0, 2 * T0, 4001)
    fig = plt.figure(figsize=(12, 6.4))
    mreza = fig.add_gridspec(2, 2, width_ratios=[1.6, 1])
    ax_t, ax_z, ax_b = (fig.add_subplot(mreza[0, 0]), fig.add_subplot(mreza[0, 1]),
                        fig.add_subplot(mreza[1, :]))
    ax_t.plot(1e3 * t, valni_oblik(val, t), color=SIVA, lw=5, alpha=0.45, label="val $x(t)$")
    ax_t.plot(1e3 * t, zbroj(val, N, t), color=BOJA_S, lw=1.8, label="$S_N$")
    ax_t.set_ylim(-1.5, 1.5)
    ax_t.set_xlabel("vrijeme $t$ / ms")
    ax_t.set_title(f"{NAZIVI[val]}: $S_N$ za $N$ = {N}", loc="left", fontsize=10)
    ax_t.legend(loc="lower left", frameon=True, framealpha=0.9, fontsize=9)

    # uvećano: uz skok (pravokutni u t = 0, pilasti u T₀/2) ili uz vrh (trokutni u T₀/4)
    sredina = {"pravokutni": 0.0, "pilasti": T0 / 2, "trokutni": T0 / 4}[val]
    tz = np.linspace(sredina - T0 / 8, sredina + T0 / 8, 2001)
    ax_z.plot(1e6 * (tz - sredina), valni_oblik(val, tz), color=SIVA, lw=5, alpha=0.45)
    ax_z.plot(1e6 * (tz - sredina), zbroj(val, N, tz), color=BOJA_S, lw=1.8)
    ax_z.set_ylim(-1.4, 1.4)
    mjesto = "uz vrh" if val == "trokutni" else "uz skok"
    ax_z.set_xlabel(f"vrijeme od mjesta {mjesto} / μs")
    ax_z.set_title(f"uvećano {mjesto}", loc="left", fontsize=10)

    spektar(ax_b, val, N_MAX, u_zbroju=N)
    ax_b.set_title("spektar: tamne linije su u zbroju", loc="left", fontsize=10)
    for ax in (ax_t, ax_z):
        ax.xaxis.set_major_formatter(zarez)
        ax.yaxis.set_major_formatter(zarez)
    fig.tight_layout()
    plt.show()
    for redak in opis(val, N):
        print(redak)
    display(Audio(zvuk(val, N), rate=FS, normalize=False))


klizac = widgets.interactive(
    pokus,
    val=widgets.ToggleButtons(options=VALOVI, value=VAL_POCETNI, description="val"),
    N=widgets.IntSlider(value=N_POCETNI, min=1, max=N_MAX, step=1, description="N",
                        continuous_update=False),
)
display(klizac)

## 4. Gibbsova pojava

Uz skok pravokutnog vala djelomični zbroj preskoči razinu vala. Teorem o Gibbsovoj granici u
poglavlju 9 pokazuje da vrh zbroja kad $N \to \infty$ ne teži razini vala $1$, nego granici

$$\frac{2}{\pi}\,\mathrm{Si}(\pi) \approx 1{,}179 ,\qquad
\mathrm{Si}(x) = \int_0^x \frac{\sin\xi}{\xi}\,d\xi ,$$

pa nadvišenje teži oko $0{,}179$, tj. oko $9\,\%$ skoka (skok je $2$, s $-1$ na $+1$). Ista
granica vrijedi uz svaki skok (pravilo o nadvišenju uz skok u poglavlju 9).

Ćelija ispod računa vrh i nadvišenje za $N$ = 11, 21 i 51, kao poglavlje 9, i uspoređuje ih s
brojkama iz knjige. Slika lijevo prikazuje te zbrojeve uz skok, a desno nadvišenje za neparne
$N$ od 11 do 101 uz granicu (crtkano).

In [ ]:
# brojke iz knjige (NADVISENJE_KNJIGA, GRANICA_KNJIGA …) su u ćeliji postavki
for redak in tablica_gibbs():
    print(redak)
for N, knjiga in NADVISENJE_KNJIGA.items():
    assert round(vrh("pravokutni", N) - 1, 4) == knjiga
g = gibbs_granica()
assert round(g, 3) == GRANICA_KNJIGA and round(g - 1, 4) == NADVISENJE_GRANICA_KNJIGA
assert round(100 * (g - 1) / SKOK, 1) == POSTO_KNJIGA
print("Bilježnica i knjiga daju iste brojke.")

T0 = 1 / F0
fig, (ax_s, ax_n) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.3, 1]})
tz = np.linspace(-T0 / 40, T0 / 8, 3001)
ax_s.plot(1e6 * tz, valni_oblik("pravokutni", tz), color=SIVA, lw=5, alpha=0.45, label="val")
for i, N in enumerate(NADVISENJE_KNJIGA):
    ax_s.plot(1e6 * tz, zbroj("pravokutni", N, tz), color=TAMNA, lw=1.0 + 0.4 * i, ls=CRTE[2 - i],
              label=f"$S_{{{N}}}$")
ax_s.axhline(g, color=TAMNA, ls=":", lw=1.2, label="granica vrha")
ax_s.set_ylim(-0.2, 1.35)
ax_s.set_xlabel("vrijeme $t$ / μs (skok u $t = 0$)")
ax_s.set_title("pravokutni val uz skok", loc="left", fontsize=10)
ax_s.legend(loc="lower right", frameon=True, framealpha=0.9, fontsize=9)

niz_N = np.arange(N_ISPIS_MIN, N_GIBBS_MAX + 1, 2)
ax_n.plot(niz_N, [vrh("pravokutni", N) - 1 for N in niz_N], "o", ms=4, mfc="white",
          mec=TAMNA, label="nadvišenje $S_N$")
ax_n.axhline(g - 1, color=TAMNA, ls="--", lw=1.2, label="granica")
ax_n.set_xlabel("broj harmonika $N$ (neparni)")
ax_n.set_ylabel("nadvišenje iznad 1")
ax_n.set_title("nadvišenje teži granici", loc="left", fontsize=10)
ax_n.legend(loc="upper right", frameon=True, framealpha=0.9, fontsize=9)
for ax in (ax_s, ax_n):
    ax.xaxis.set_major_formatter(zarez)
    ax.yaxis.set_major_formatter(zarez)
fig.tight_layout()
plt.show()

## 5. Slušanje niza djelomičnih zbrojeva

Za svaki val bilježnica složi zvukove $S_N$ za $N$ = 1, 7, 21 i 51, jedan za drugim, s kratkom
stankom. Usporedite kako se mijenja boja tona i kako se tri vala razlikuju uz isti $N$.

In [ ]:
NIZ_N = (*N_SLIKA, N_MAX)
for val in VALOVI:
    redak = opis_niza(val, NIZ_N)
    print(redak)
    display(Audio(niz_zvukova(val, NIZ_N), rate=FS, normalize=False))

## 6. Faza: isti amplitudni spektar, drukčiji oblik

Zbrojimo prvih $N = 21$ harmonika pravokutnog vala s istim amplitudama, ali svakom harmoniku
dodajmo slučajnu fazu $\varphi_k$:

$$\tilde{S}_N(t) = \sum_{k=1}^{N} b_k \sin(k\omega_0 t + \varphi_k) .$$

Amplitudni spektar $|b_k|$ je isti, pa je prema Parsevalovoj jednakosti (poglavlje 9) ista i
snaga. Oblik vala je uz zadano sjeme (`SJEME`) posve drukčiji: na grafu nema ravnih dijelova
kao kod $S_N$. Oba zvuka imaju istu efektivnu vrijednost. Poslušajte ih: zvuče slično, ali ne
nužno istovjetno, iako im se oblici jako razlikuju. Koliko se razlikuju, ovisi o osnovnoj
frekvenciji (zadatak 4 u odjeljku *Pokušajte sami*).

In [ ]:
rng = np.random.default_rng(SJEME)
faze = rng.uniform(0, 2 * np.pi, N_FAZE)
t = np.linspace(0, 2 / F0, 4001)
fig, (ax_t, ax_b) = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1.6, 1]})
ax_t.plot(1e3 * t, zbroj("pravokutni", N_FAZE, t), color=TAMNA, lw=1.5, label="faze 0: $S_N$")
ax_t.plot(1e3 * t, zbroj("pravokutni", N_FAZE, t, faze), color=BOJA_S, lw=1.5, ls="--",
          label="slučajne faze")
ax_t.set_xlabel("vrijeme $t$ / ms")
ax_t.set_title(f"pravokutni val, $N$ = {N_FAZE}: isti harmonici, druge faze", loc="left",
               fontsize=10)
ax_t.legend(loc="upper left", bbox_to_anchor=(0, -0.18), ncol=2, frameon=False, fontsize=9)
k = np.arange(1, N_FAZE + 1)
amplitude = np.array([abs(koeficijent("pravokutni", kk)) for kk in k])
ax_b.vlines(k, 0, amplitude, color=TAMNA, lw=1.6)
ax_b.plot(k, amplitude, "o", color=TAMNA, mfc="white", ms=5)
ax_b.set_xlabel("harmonik $k$")
ax_b.set_ylabel("$|b_k|$")
ax_b.xaxis.set_major_locator(MaxNLocator(integer=True, steps=[1, 2, 5, 10]))
ax_b.set_title("amplitudni spektar, isti za oba zbroja", loc="left", fontsize=10)
for ax in (ax_t, ax_b):
    ax.xaxis.set_major_formatter(zarez)
    ax.yaxis.set_major_formatter(zarez)
fig.tight_layout()
plt.show()

# ista snaga (isti |b_k|), pa isti faktor za oba zvuka; vrh najviše 0,9
s_nula = zvuk("pravokutni", N_FAZE, glasnoca=0.3)
s_faze = zvuk("pravokutni", N_FAZE, faze, glasnoca=0.3)
faktor = 0.9 / max(np.max(np.abs(s_nula)), np.max(np.abs(s_faze)))
print("Faze 0 (djelomični zbroj pravokutnog vala):")
display(Audio(faktor * s_nula, rate=FS, normalize=False))
print("Iste amplitude, slučajne faze:")
display(Audio(faktor * s_faze, rate=FS, normalize=False))

## Pokušajte sami

1. Poslušajte pravokutni, pilasti i trokutni val uz isti $N$, npr. $N = 21$. Koji zvuči
   najmekše? Usporedite njihove spektre i povežite razliku s pravilom o glatkoći i padu
   koeficijenata iz poglavlja 9.
2. Pilasti val ima sve harmonike, a pravokutni i trokutni samo neparne. Usporedite pilasti i
   pravokutni val na sluh uz isti $N$. Kako biste opisali razliku u boji tona?
3. U postavkama zadajte `F0 = 110.0`, a zatim `F0 = 880.0`, i pokrenite sve ćelije. Uz koji
   $N$ se zvuk pravokutnog vala za vaše uho prestane mijenjati? Kako to ovisi o $f_0$ i o
   najvišoj frekvenciji koju čujete?
4. U pokusu s fazama zadajte drugo `SJEME`, a zatim i nižu osnovnu frekvenciju, npr.
   `F0 = 55.0`. Čuje li se sada razlika između dvaju zvukova? Pokušajte objasniti zašto.
5. Dopišite ćeliju koja koeficijente pravokutnog vala množi težinama
   $w_k = 1 - \dfrac{k - 1}{N}$, pa nacrtajte zbroj uz skok i usporedite ga s $S_N$ za isti
   $N$. Što se dogodi s nadvišenjem, a što sa strminom zbroja uz skok? Kako se to čuje?

---
Kod je pod licencom MIT (`LICENSE`), a tekst pod CC BY 4.0 (`LICENSE-sadrzaj.md`).